# VietFinQA — Local Metrics

## Bộ metric chính

1. Exact Match
2. Token F1
3. ROUGE-L F1
4. BERTScore F1 (PhoBERT)
5. Numerical F1
6. Named-Entity F1
7. Grounding Precision


## 1. Cài đặt và import

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("tqdm", "tqdm"),
    ("matplotlib", "matplotlib"),
    ("rouge-score", "rouge_score"),
    ("bert-score", "bert_score"),
    ("transformers", "transformers"),
    ("torch", "torch"),
    ("underthesea", "underthesea"),
]

for install_name, import_name in PACKAGES:
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {install_name} ...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", install_name
        ])

print("Dependencies are ready.")


In [ ]:

import json
import os
import glob
import re
import unicodedata
import warnings
from collections import Counter
from decimal import Decimal, InvalidOperation
from typing import Any, Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("default")

try:
    import torch
    DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEFAULT_DEVICE = "cpu"

print(f"Default device: {DEFAULT_DEVICE}")


## 2. Cấu hình

In [ ]:
# Dùng chính thư mục đang chạy notebook, nơi chứa các file prediction CSV.
DATA_DIR = os.path.abspath(os.getcwd())

MODEL_FILES = {
    "SCS LoRA gold test - LLaMA 3.2 1B":
        "llama_3_2_1b_scs_lora_gold_test_predictions.csv",
    "SCS LoRA gold test - Qwen3 0.6B":
        "qwen3_0_6b_scs_lora_gold_test_predictions.csv",
    "Question-type mixture LoRA soft routing gold test - Qwen3 1.7B":
        "qwen3_1_7b_question_type_mixture_lora_soft_routing_gold_test_predictions.csv",
    "SCS LoRA no prompt gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_no_prompt_gold_test_predictions.csv",
    "SCS LoRA prompt scale 0.5 r16 a32 gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_prompt_scale_0_5_r16_a32_gold_test_predictions.csv",
    "SCS LoRA prompt scale 0.5 r4 a8 gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_prompt_scale_0_5_r4_a8_gold_test_predictions.csv",
    "SCS LoRA prompt scale 0.5 gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_prompt_scale_0_5_gold_test_predictions.csv",
    "SCS LoRA prompt scale 2.0 gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_prompt_scale_2_0_gold_test_predictions.csv",
    "SCS LoRA gold test - Qwen3 1.7B":
        "qwen3_1_7b_scs_lora_gold_test_predictions.csv",
}

# Khi một số file chưa được upload, notebook bỏ qua file đó và vẫn đánh giá các file còn lại.
SKIP_MISSING_FILES = True

REFERENCE_COLUMN = "answer"
PREDICTION_COLUMN = "prediction"
QUESTION_TYPE_COLUMN = "gold_question_type"

# Grounding của prediction được kiểm tra với context retrieve.
CONTEXT_COLUMN = "context_used"

RUN_ROUGE = True
RUN_BERTSCORE = True
RUN_NER = True

METRIC_COLUMNS = [
    "exact_match",
    "token_f1",
    "rouge_l_f1",
    "bertscore_f1",
    "num_f1",
    "entity_f1",
    "grounding_precision",
]

# Cấu hình giống notebook eval_llama3_2_1b_merge.ipynb.
BERTSCORE_MODEL = "vinai/phobert-base"
BERTSCORE_LANG = "vi"
BERTSCORE_BATCH_SIZE = 32
BERTSCORE_USE_FAST_TOKENIZER = True
BERTSCORE_FALLBACK_MULTILINGUAL = True

OUTPUT_DIR = os.path.join(DATA_DIR, "eval_results_selected_metrics")
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 3. Chuẩn hóa văn bản và metric QA tổng quát

In [ ]:
import re
import unicodedata
from collections import Counter
from decimal import Decimal, InvalidOperation
from typing import Dict, List, Optional, Set, Tuple, Any

import numpy as np
import pandas as pd

def safe_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)

def normalize_text(text: Any) -> str:
    text = unicodedata.normalize("NFC", safe_text(text)).lower()
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = re.sub(r"[_]+", " ", text)
    text = re.sub(r"[^\w%.,/+:-]+", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _parse_decimal_surface(token: str, decimal_hint: bool = False) -> Optional[Decimal]:
    token = token.strip().replace(" ", "").replace("−", "-")
    if not token:
        return None
    sign = ""
    if token[0] in "+-":
        sign, token = token[0], token[1:]
    if not token or not re.fullmatch(r"\d+(?:[.,]\d+)*", token):
        return None

    if "." in token and "," in token:
        last_dot = token.rfind(".")
        last_comma = token.rfind(",")
        decimal_sep = "." if last_dot > last_comma else ","
        thousands_sep = "," if decimal_sep == "." else "."
        token = token.replace(thousands_sep, "")
        token = token.replace(decimal_sep, ".")
    elif "." in token or "," in token:
        sep = "." if "." in token else ","
        parts = token.split(sep)
        if len(parts) > 2:
            # repeated separator is normally a thousands grouping
            token = "".join(parts)
        else:
            left, right = parts
            if decimal_hint:
                token = left + "." + right
            elif len(right) == 3 and len(left) >= 1:
                token = left + right
            else:
                token = left + "." + right

    try:
        return Decimal(sign + token)
    except InvalidOperation:
        return None

def _decimal_key(value: Decimal) -> str:
    if value == 0:
        return "0"
    s = format(value.normalize(), "f")
    if "." in s:
        s = s.rstrip("0").rstrip(".")
    return s

_TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)*(?:%)?|[^\W\d_]+", re.UNICODE)

def qa_tokens(text: Any) -> List[str]:
    text = normalize_text(text)
    tokens = []
    for token in _TOKEN_RE.findall(text):
        if re.match(r"^\d", token):
            is_pct = token.endswith("%")
            raw = token[:-1] if is_pct else token
            value = _parse_decimal_surface(raw, decimal_hint=is_pct)
            if value is not None:
                token = _decimal_key(value) + ("%" if is_pct else "")
        tokens.append(token)
    return tokens

def exact_match(reference: Any, prediction: Any) -> float:
    return float(qa_tokens(reference) == qa_tokens(prediction))

def token_f1(reference: Any, prediction: Any) -> float:
    ref = qa_tokens(reference)
    pred = qa_tokens(prediction)
    if not ref and not pred:
        return 1.0
    if not ref or not pred:
        return 0.0
    common = Counter(ref) & Counter(pred)
    matched = sum(common.values())
    precision = matched / len(pred)
    recall = matched / len(ref)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


## 4. Metric số có giữ đơn vị

In [ ]:
_DATE_RE = re.compile(r"(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})(?!\d)")

_UNIT_RE = (
    r"%(?!\w)"
    r"|điểm\s+phần\s+trăm"
    r"|điểm\s+cơ\s+bản"
    r"|bps"
    r"|(?:nghìn|ngàn)\s+tỷ(?:\s+(?:đồng|vnd|usd|eur))?"
    r"|tỷ(?:\s+(?:đồng|vnd|usd|eur))?"
    r"|triệu(?:\s+(?:đồng|vnd|usd|eur))?"
    r"|(?:nghìn|ngàn)(?:\s+(?:đồng|vnd|usd|eur))?"
    r"|(?:đồng|vnd|usd|eur)"
)
_NUMBER_RE = re.compile(
    rf"(?<![\w/])(?P<number>[+-]?\d+(?:[.,]\d+)*)(?:\s*(?P<unit>{_UNIT_RE}))?(?![\w/])",
    re.IGNORECASE | re.UNICODE,
)

def _normalise_unit(unit: str) -> Tuple[str, Decimal, bool]:
    u = re.sub(r"\s+", " ", (unit or "").strip().lower())
    if u == "%":
        return "percent", Decimal("1"), True
    if u == "điểm phần trăm":
        return "percentage_point", Decimal("1"), True
    if u in {"điểm cơ bản", "bps"}:
        return "percentage_point", Decimal("0.01"), True

    scale = Decimal("1")
    if re.search(r"(?:nghìn|ngàn)\s+tỷ", u):
        scale = Decimal("1000000000000")
    elif re.search(r"\btỷ\b", u):
        scale = Decimal("1000000000")
    elif re.search(r"\btriệu\b", u):
        scale = Decimal("1000000")
    elif re.search(r"\b(?:nghìn|ngàn)\b", u):
        scale = Decimal("1000")

    if re.search(r"\b(?:đồng|vnd)\b", u):
        kind = "vnd"
    elif re.search(r"\busd\b", u):
        kind = "usd"
    elif re.search(r"\beur\b", u):
        kind = "eur"
    elif scale != 1:
        kind = "scaled"
    else:
        kind = "plain"
    return kind, scale, bool(u)

def extract_numeric_facts(text: Any) -> Set[Tuple[str, str]]:
    """
    Extract unit-aware numerical facts.
    Examples:
      1.500 tỷ đồng -> ('vnd', '1500000000000')
      1,5 tỷ đồng  -> ('vnd', '1500000000')
      50 bps       -> ('percentage_point', '0.5')
      22/08/2023   -> ('date', '2023-08-22')
    """
    text = normalize_text(text)
    facts: Set[Tuple[str, str]] = set()
    chars = list(text)

    for match in _DATE_RE.finditer(text):
        day, month, year = map(int, match.groups())
        if year < 100:
            year += 2000 if year < 50 else 1900
        if 1 <= month <= 12 and 1 <= day <= 31:
            facts.add(("date", f"{year:04d}-{month:02d}-{day:02d}"))
            for i in range(match.start(), match.end()):
                chars[i] = " "

    masked = "".join(chars)
    for match in _NUMBER_RE.finditer(masked):
        raw_num = match.group("number")
        raw_unit = match.group("unit") or ""
        kind, multiplier, has_unit = _normalise_unit(raw_unit)
        decimal_hint = kind in {"percent", "percentage_point"}
        value = _parse_decimal_surface(raw_num, decimal_hint=decimal_hint)
        if value is None:
            continue
        value *= multiplier
        facts.add((kind, _decimal_key(value)))
    return facts

def _fact_equivalent(a: Tuple[str, str], b: Tuple[str, str]) -> bool:
    kind_a, value_a = a
    kind_b, value_b = b
    if value_a != value_b:
        return False
    if kind_a == kind_b:
        return True
    # A scaled value may omit an explicit currency marker in one answer.
    return {kind_a, kind_b} <= {"scaled", "vnd"} or {kind_a, kind_b} <= {"scaled", "usd"} or {kind_a, kind_b} <= {"scaled", "eur"}

def _greedy_fact_matches(
    reference_facts: Set[Tuple[str, str]],
    prediction_facts: Set[Tuple[str, str]],
) -> List[Tuple[Tuple[str, str], Tuple[str, str]]]:
    remaining = set(prediction_facts)
    matched = []
    for ref_fact in reference_facts:
        candidate = next((p for p in remaining if _fact_equivalent(ref_fact, p)), None)
        if candidate is not None:
            matched.append((ref_fact, candidate))
            remaining.remove(candidate)
    return matched

def numerical_f1(reference: Any, prediction: Any) -> float:
    ref_facts = extract_numeric_facts(reference)
    pred_facts = extract_numeric_facts(prediction)
    matches = _greedy_fact_matches(ref_facts, pred_facts)
    matched = len(matches)

    # Full-data convention: every row receives a numerical score.
    if not ref_facts and not pred_facts:
        return 1.0

    if not ref_facts and pred_facts:
        return 0.0

    precision = matched / len(pred_facts) if pred_facts else 0.0
    recall = matched / len(ref_facts)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def numerical_grounding(prediction: Any, context: Any) -> Dict[str, Any]:
    pred_facts = extract_numeric_facts(prediction)
    ctx_facts = extract_numeric_facts(context)
    if not pred_facts:
        return {"precision": 1.0, "pred_count": 0, "supported_count": 0}
    if not safe_text(context).strip():
        return {
            "precision": 0.0,
            "pred_count": len(pred_facts),
            "supported_count": 0,
        }
    matches = _greedy_fact_matches(pred_facts, ctx_facts)
    supported = len(matches)
    precision = supported / len(pred_facts)
    return {
        "precision": precision,
        "pred_count": len(pred_facts),
        "supported_count": supported,
    }


### Quy ước Numerical F1 trên toàn bộ dữ liệu

- So khớp theo **giá trị đã chuẩn hóa + loại đơn vị**.
- Hỗ trợ ngày `dd/mm/yyyy`, `dd-mm-yyyy`, `%`, điểm phần trăm, điểm cơ bản, VND, USD, EUR và các hệ số `nghìn/ngàn`, `triệu`, `tỷ`, `nghìn/ngàn tỷ`.
- Reference và prediction đều không có số: `Numerical F1 = 1.0`.
- Reference không có số nhưng prediction sinh thêm số: `Numerical F1 = 0.0`.
- Reference có số nhưng prediction không có số: `Numerical F1 = 0.0`.

Quy ước này cho phép lấy trung bình Numerical F1 trên toàn bộ số dòng của file, nhưng score có thể chịu ảnh hưởng bởi tỷ lệ câu không chứa số. Vì vậy nên đọc cùng các metric tổng quát như Token F1, ROUGE-L và PhoBERTScore.

## 5. Metric thực thể tùy chọn

In [ ]:
_NER_CACHE: Dict[str, Set[str]] = {}

def normalize_entity(text: Any) -> str:
    value = unicodedata.normalize("NFC", safe_text(text)).lower().replace("_", " ")
    value = re.sub(r"[^\w\s-]+", " ", value, flags=re.UNICODE)
    return re.sub(r"\s+", " ", value).strip(" -")

def get_entities(text: Any) -> Set[str]:
    raw = safe_text(text)
    if raw in _NER_CACHE:
        return _NER_CACHE[raw]
    if not raw.strip():
        _NER_CACHE[raw] = set()
        return set()
    try:
        from underthesea import ner
        tagged = ner(raw)
        entities: Set[str] = set()
        current: List[str] = []
        current_type: Optional[str] = None

        for item in tagged:
            word = str(item[0])
            tag = str(item[3]) if len(item) >= 4 else "O"
            if tag.startswith("B-"):
                if current:
                    entity = normalize_entity(" ".join(current))
                    if entity:
                        entities.add(entity)
                current = [word]
                current_type = tag[2:]
            elif tag.startswith("I-") and current_type:
                current.append(word)
            else:
                if current:
                    entity = normalize_entity(" ".join(current))
                    if entity:
                        entities.add(entity)
                current = []
                current_type = None

        if current:
            entity = normalize_entity(" ".join(current))
            if entity:
                entities.add(entity)

        _NER_CACHE[raw] = entities
        return entities
    except Exception as exc:
        warnings.warn(f"NER unavailable for one sample: {exc}")
        _NER_CACHE[raw] = set()
        return set()

def _entity_equivalent(a: str, b: str) -> bool:
    if a == b:
        return True
    if min(len(a), len(b)) < 4:
        return False
    return a in b or b in a

def _greedy_entity_matches(ref_entities: Set[str], pred_entities: Set[str]) -> List[Tuple[str, str]]:
    remaining = set(pred_entities)
    matched = []
    for ref_entity in ref_entities:
        candidate = next((p for p in remaining if _entity_equivalent(ref_entity, p)), None)
        if candidate is not None:
            matched.append((ref_entity, candidate))
            remaining.remove(candidate)
    return matched

def _entity_in_text(entity: str, text: Any) -> bool:
    entity = normalize_entity(entity)
    haystack = normalize_entity(text)
    if not entity or not haystack:
        return False
    return f" {entity} " in f" {haystack} "

def entity_f1(reference: Any, prediction: Any) -> float:
    ref_entities = get_entities(reference)
    pred_entities = get_entities(prediction)
    matches = _greedy_entity_matches(ref_entities, pred_entities)
    matched = len(matches)

    if not ref_entities and not pred_entities:
        return 1.0

    if not ref_entities and pred_entities:
        return 0.0

    precision = matched / len(pred_entities) if pred_entities else 0.0
    recall = matched / len(ref_entities)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def entity_grounding(prediction: Any, context: Any) -> Dict[str, Any]:
    pred_entities = get_entities(prediction)
    if not pred_entities:
        return {"precision": 1.0, "pred_count": 0, "supported_count": 0}
    if not safe_text(context).strip():
        return {
            "precision": 0.0,
            "pred_count": len(pred_entities),
            "supported_count": 0,
        }
    supported_entities = {e for e in pred_entities if _entity_in_text(e, context)}
    return {
        "precision": len(supported_entities) / len(pred_entities),
        "pred_count": len(pred_entities),
        "supported_count": len(supported_entities),
    }


`Named-Entity F1` phụ thuộc vào chất lượng NER của `underthesea` và được tính trên toàn bộ dữ liệu theo cùng quy ước với Numerical F1:

- reference và prediction đều không có entity: score = `1.0`;
- prediction sinh entity khi reference không có entity: score = `0.0`;
- reference có entity nhưng prediction không có entity: score = `0.0`.

Có thể đặt `RUN_NER = False` để bỏ toàn bộ metric thực thể.

## 6. Grounding local


In [ ]:
def combine_grounding(
    numerical: Dict[str, Any],
    entity: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    num_pred = int(numerical.get("pred_count", 0))
    num_supported = int(numerical.get("supported_count", 0))

    ent_pred = int(entity.get("pred_count", 0)) if entity is not None else 0
    ent_supported = int(entity.get("supported_count", 0)) if entity is not None else 0

    total_pred = num_pred + ent_pred
    total_supported = num_supported + ent_supported

    if total_pred == 0:
        return {"precision": 1.0}

    return {"precision": total_supported / total_pred}


`grounding_precision` kết hợp numerical facts và named entities theo số fact thực tế trong prediction, rồi kiểm tra bằng chứng bề mặt trong `context_used`.

Khi prediction không chứa fact số hoặc thực thể, grounding precision được quy ước là `1.0`. Đây là proxy local dựa trên khớp bề mặt, không phải kiểm chứng ngữ nghĩa đầy đủ.


## 7. ROUGE và BERTScore

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn


_ROUGE_SCORER = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False,
)


def compute_rouge_batch(
    references: List[str],
    predictions: List[str],
    description: str = "ROUGE",
) -> List[Optional[float]]:
    """Compute per-row ROUGE-L F1 only."""
    if len(references) != len(predictions):
        raise ValueError("references and predictions must have the same length")

    output: List[Optional[float]] = []

    for reference, prediction in tqdm(
        zip(references, predictions),
        total=len(references),
        desc=description,
        leave=False,
    ):
        reference_text = safe_text(reference).strip()
        prediction_text = safe_text(prediction).strip()

        if not reference_text and not prediction_text:
            output.append(1.0)
            continue

        if not reference_text or not prediction_text:
            output.append(0.0)
            continue

        try:
            score = _ROUGE_SCORER.score(
                reference_text,
                prediction_text,
            )["rougeL"]
            output.append(float(score.fmeasure))
        except Exception as exc:
            warnings.warn(
                f"{description}: one row was skipped because of: {exc}"
            )
            output.append(None)

    return output


def compute_bertscore_batch(
    references: List[str],
    predictions: List[str],
    model_type: str,
    lang: str,
    batch_size: int,
    use_fast_tokenizer: bool,
    fallback_multilingual: bool = True,
) -> List[Optional[float]]:
    """
    Compute per-row BERTScore F1 with bert_score.score.
    """
    if len(references) != len(predictions):
        raise ValueError("references and predictions must have the same length")

    if not references:
        return []

    predictions_clean = [
        safe_text(text).strip() or "[UNK]"
        for text in predictions
    ]
    references_clean = [
        safe_text(text).strip() or "[UNK]"
        for text in references
    ]

    try:
        _precision, _recall, f1 = bert_score_fn(
            predictions_clean,
            references_clean,
            model_type=model_type,
            lang=lang,
            batch_size=max(1, int(batch_size)),
            verbose=False,
            use_fast_tokenizer=bool(use_fast_tokenizer),
        )
    except Exception as first_exc:
        if not fallback_multilingual:
            warnings.warn(
                f"PhoBERT BERTScore was skipped because of: {first_exc}"
            )
            return [None] * len(predictions)

        warnings.warn(
            "PhoBERT BERTScore failed; retrying with the multilingual "
            f"bert-score model for lang={lang!r}. Reason: {first_exc}"
        )
        try:
            _precision, _recall, f1 = bert_score_fn(
                predictions_clean,
                references_clean,
                lang=lang,
                batch_size=max(1, int(batch_size)),
                verbose=False,
                use_fast_tokenizer=bool(use_fast_tokenizer),
            )
        except Exception as fallback_exc:
            warnings.warn(
                "BERTScore was skipped because both PhoBERT and multilingual "
                f"fallback failed. PhoBERT error: {first_exc}. "
                f"Fallback error: {fallback_exc}"
            )
            return [None] * len(predictions)

    return [
        float(value)
        for value in f1.detach().cpu().tolist()
    ]


## 8. Đọc dữ liệu và đánh giá

In [ ]:
REQUIRED_COLUMNS = {
    REFERENCE_COLUMN,
    PREDICTION_COLUMN,
    CONTEXT_COLUMN,
}


def validate_dataframe(df: pd.DataFrame, model_name: str) -> None:
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(
            f"{model_name}: missing required columns: {sorted(missing)}"
        )


def build_sample_id(df: pd.DataFrame) -> pd.Series:
    # Ưu tiên ID gốc có sẵn trong file.
    for id_column in ["sample_id", "qa_id", "example_id", "row_id"]:
        if id_column in df.columns:
            return df[id_column].map(safe_text)

    if "url" in df.columns and "question" in df.columns:
        base = (
            df["url"].map(safe_text)
            + " || "
            + df["question"].map(safe_text)
        )
        duplicate_rank = base.groupby(base).cumcount().astype(str)
        return base + " || duplicate=" + duplicate_rank

    return pd.Series(
        [f"row={i}" for i in range(len(df))],
        index=df.index,
        dtype="object",
    )


def resolve_prediction_path(file_name: str) -> Optional[str]:
    """Tìm file chuẩn hoặc bản có hậu tố trình duyệt như (1), (2)."""
    candidates = []
    if os.path.isabs(file_name):
        candidates.append(file_name)
    else:
        candidates.extend([os.path.join(DATA_DIR, file_name), file_name])

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    stem, extension = os.path.splitext(file_name)
    search_dirs = (
        [DATA_DIR, "."]
        if not os.path.isabs(file_name)
        else [os.path.dirname(file_name)]
    )
    for directory in search_dirs:
        if not os.path.isdir(directory):
            continue
        pattern = os.path.join(directory, f"{stem}(*){extension}")
        numbered_matches = sorted(glob.glob(pattern))
        if numbered_matches:
            return numbered_matches[0]

    return None


def evaluate_sample(row: pd.Series, run_ner: bool) -> Dict[str, Any]:
    reference = safe_text(row.get(REFERENCE_COLUMN, ""))
    prediction = safe_text(row.get(PREDICTION_COLUMN, ""))
    context = safe_text(row.get(CONTEXT_COLUMN, ""))

    tf1 = token_f1(reference, prediction)
    nf1 = numerical_f1(reference, prediction)
    num_ground = numerical_grounding(prediction, context)

    if run_ner:
        ef1 = entity_f1(reference, prediction)
        ent_ground = entity_grounding(prediction, context)
    else:
        ef1 = np.nan
        ent_ground = None

    fact_ground = combine_grounding(num_ground, ent_ground)

    return {
        "exact_match": exact_match(reference, prediction),
        "token_f1": tf1,

        # Hai metric này được điền theo batch sau khi nối các file.
        "rouge_l_f1": np.nan,
        "bertscore_f1": np.nan,

        "num_f1": nf1,
        "entity_f1": ef1,
        "grounding_precision": fact_ground["precision"],
    }


def evaluate_model_file(
    model_name: str,
    csv_path: str,
    run_ner: bool,
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    validate_dataframe(df, model_name)
    df = df.copy()
    print(
        f"Loaded {len(df):,} rows; "
        "all rows will be evaluated (no sampling)."
    )
    df["sample_id"] = build_sample_id(df)

    records = []
    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Local metrics: {model_name}",
    ):
        records.append(evaluate_sample(row, run_ner=run_ner))

    score_columns = list(METRIC_COLUMNS)

    scores = pd.DataFrame([
        {
            key: record.get(key, np.nan)
            for key in score_columns
        }
        for record in records
    ])

    scores.insert(0, "model", model_name)
    scores.insert(1, "sample_id", df["sample_id"].values)

    for column in [
        "url",
        "title",
        "time",
        CONTEXT_COLUMN,
        "question",
        REFERENCE_COLUMN,
        PREDICTION_COLUMN,
        QUESTION_TYPE_COLUMN,
    ]:
        if column in df.columns and column not in scores.columns:
            scores[column] = df[column].values

    return scores


all_score_frames = []
resolved_model_files = {}
missing_model_files = {}

for model_name, file_name in MODEL_FILES.items():
    csv_path = resolve_prediction_path(file_name)
    if csv_path is None:
        missing_model_files[model_name] = file_name
        message = (
            f"File not found, skipped: "
            f"{model_name} -> {file_name}"
        )
        if SKIP_MISSING_FILES:
            warnings.warn(message)
            continue
        raise FileNotFoundError(message)

    resolved_model_files[model_name] = csv_path
    print()
    print(f"Evaluating {model_name}: {csv_path}")
    model_scores = evaluate_model_file(
        model_name=model_name,
        csv_path=csv_path,
        run_ner=RUN_NER,
    )
    all_score_frames.append(model_scores)

if not all_score_frames:
    raise FileNotFoundError(
        "No prediction file was found. "
        "Check DATA_DIR and MODEL_FILES."
    )

df_scores = pd.concat(all_score_frames, ignore_index=True)
print()
print(
    f"Local metrics completed: {len(df_scores):,} "
    f"model-sample rows from {len(all_score_frames)} file(s)."
)

# ------------------------------------------------------------
# Answer ROUGE-L F1 through rouge-score
# ------------------------------------------------------------
if RUN_ROUGE:
    print("\nComputing answer ROUGE-L F1 with rouge-score ...")
    df_scores["rouge_l_f1"] = compute_rouge_batch(
        references=df_scores[REFERENCE_COLUMN].map(safe_text).tolist(),
        predictions=df_scores[PREDICTION_COLUMN].map(safe_text).tolist(),
        description="Answer ROUGE",
    )

# ------------------------------------------------------------
# Answer BERTScore F1 through bert_score.score
# ------------------------------------------------------------
if RUN_BERTSCORE:
    print("\nComputing BERTScore F1 with bert_score.score ...")
    df_scores["bertscore_f1"] = compute_bertscore_batch(
        references=df_scores[REFERENCE_COLUMN].map(safe_text).tolist(),
        predictions=df_scores[PREDICTION_COLUMN].map(safe_text).tolist(),
        model_type=BERTSCORE_MODEL,
        lang=BERTSCORE_LANG,
        batch_size=BERTSCORE_BATCH_SIZE,
        use_fast_tokenizer=BERTSCORE_USE_FAST_TOKENIZER,
        fallback_multilingual=BERTSCORE_FALLBACK_MULTILINGUAL,
    )
print()
print(
    f"Completed: {len(df_scores):,} model-sample rows "
    f"from {len(all_score_frames)} file(s)."
)
if missing_model_files:
    print(f"Skipped {len(missing_model_files)} missing file(s).")

display(df_scores.head())


## 10. Tổng hợp metric trên toàn bộ dữ liệu

Bảng tổng hợp lấy trung bình của 7 metric trên toàn bộ các dòng trong từng file.


In [ ]:
METRIC_LABELS = {
    "exact_match": "Exact Match",
    "token_f1": "Token F1",
    "rouge_l_f1": "ROUGE-L F1",
    "bertscore_f1": "BERTScore F1 (PhoBERT)",
    "num_f1": "Numerical F1",
    "entity_f1": "Named-Entity F1",
    "grounding_precision": "Grounding Precision",
}

HIGHER_IS_BETTER = {metric: True for metric in METRIC_LABELS}


def summarize_metrics(
    scores: pd.DataFrame,
    group_columns: List[str],
) -> pd.DataFrame:
    rows = []

    grouped = scores.groupby(group_columns, dropna=False)
    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        group_values = dict(zip(group_columns, group_key))

        for metric, label in METRIC_LABELS.items():
            if metric not in group.columns:
                continue

            values = pd.to_numeric(
                group[metric],
                errors="coerce",
            )
            mean = (
                float(values.mean())
                if values.notna().any()
                else np.nan
            )

            rows.append({
                **group_values,
                "metric": metric,
                "metric_label": label,
                "mean": mean,
                "higher_is_better": HIGHER_IS_BETTER[metric],
            })

    return pd.DataFrame(rows)


overall_summary = summarize_metrics(df_scores, ["model"])
display(
    overall_summary[
        ["model", "metric_label", "mean"]
    ].sort_values(["model", "metric_label"])
)


## 11. Bảng kết quả tổng quát

In [ ]:
overall_wide = overall_summary.pivot(
    index="model",
    columns="metric_label",
    values="mean",
)

display(overall_wide.round(4))

## 12. Phân tích theo question type bằng cùng bộ metric

In [ ]:
if QUESTION_TYPE_COLUMN in df_scores.columns:
    df_scores[QUESTION_TYPE_COLUMN] = (
        df_scores[QUESTION_TYPE_COLUMN]
        .map(safe_text)
        .str.strip()
        .str.upper()
        .replace("", "UNKNOWN")
    )

    per_type_summary = summarize_metrics(
        df_scores,
        ["model", QUESTION_TYPE_COLUMN],
    )

    per_type_mean = per_type_summary.pivot_table(
        index=["model", QUESTION_TYPE_COLUMN],
        columns="metric_label",
        values="mean",
    )

    display(per_type_mean.round(4))
else:
    per_type_summary = pd.DataFrame()
    print(f"Column '{QUESTION_TYPE_COLUMN}' is absent; per-type analysis was skipped.")

## 13. Biểu đồ tổng quát

In [ ]:
PLOT_METRICS = list(METRIC_COLUMNS)

plot_data = overall_summary[
    overall_summary["metric"].isin(PLOT_METRICS)
].copy()

for metric in PLOT_METRICS:
    subset = plot_data[
        plot_data["metric"] == metric
    ].dropna(subset=["mean"])

    if subset.empty:
        continue

    fig, ax = plt.subplots(
        figsize=(max(6, len(subset) * 1.4), 4)
    )
    x = np.arange(len(subset))
    means = subset["mean"].to_numpy(dtype=float)

    ax.bar(x, means, width=0.65)
    ax.set_xticks(x)
    ax.set_xticklabels(
        subset["model"],
        rotation=15,
        ha="right",
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(METRIC_LABELS[metric])
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(
        OUTPUT_DIR,
        f"{metric}.png",
    )
    plt.savefig(
        plot_path,
        dpi=160,
        bbox_inches="tight",
    )
    plt.show()


## 14. Error analysis không dùng luật theo question type

In [ ]:
def show_low_scoring_samples(
    scores: pd.DataFrame,
    model_name: str,
    metric: str = "token_f1",
    n: int = 10,
) -> pd.DataFrame:
    if metric not in scores.columns:
        raise KeyError(metric)

    subset = scores[
        scores["model"] == model_name
    ].copy()
    subset = (
        subset
        .dropna(subset=[metric])
        .sort_values(metric)
        .head(n)
    )

    columns = [
        "model",
        QUESTION_TYPE_COLUMN,
        "question",
        REFERENCE_COLUMN,
        PREDICTION_COLUMN,
        metric,
    ] + [name for name in METRIC_COLUMNS if name != metric]
    columns = [
        column
        for column in columns
        if column in subset.columns
    ]
    return subset[columns]


# display(show_low_scoring_samples(df_scores, "model_name", "token_f1", n=10))


## 15. Export

In [ ]:
per_sample_path = os.path.join(
    OUTPUT_DIR,
    "per_sample_scores.csv",
)
overall_path = os.path.join(
    OUTPUT_DIR,
    "overall_summary.csv",
)
overall_wide_path = os.path.join(
    OUTPUT_DIR,
    "overall_summary_wide.csv",
)
per_type_path = os.path.join(
    OUTPUT_DIR,
    "per_question_type_summary.csv",
)
df_scores.to_csv(
    per_sample_path,
    index=False,
    encoding="utf-8-sig",
)
overall_summary.to_csv(
    overall_path,
    index=False,
    encoding="utf-8-sig",
)
overall_wide.to_csv(
    overall_wide_path,
    encoding="utf-8-sig",
)

if not per_type_summary.empty:
    per_type_summary.to_csv(
        per_type_path,
        index=False,
        encoding="utf-8-sig",
    )

metadata = {
    "model_files": MODEL_FILES,
    "resolved_model_files": resolved_model_files,
    "missing_model_files": missing_model_files,
    "evaluation_scope": "all_rows_in_each_file",
    "summary_method": "arithmetic_mean_without_bootstrap",
    "reference_column": REFERENCE_COLUMN,
    "prediction_column": PREDICTION_COLUMN,
    "grounding_context_column": CONTEXT_COLUMN,
    "run_rouge": RUN_ROUGE,
    "run_bertscore": RUN_BERTSCORE,
    "run_ner": RUN_NER,
    "rouge_backend": "rouge_score.rouge_scorer.RougeScorer",
    "bertscore_backend": "bert_score.score",
    "bertscore_model": (
        BERTSCORE_MODEL
        if RUN_BERTSCORE
        else None
    ),
    "bertscore_lang": (
        BERTSCORE_LANG
        if RUN_BERTSCORE
        else None
    ),
    "bertscore_batch_size": (
        BERTSCORE_BATCH_SIZE
        if RUN_BERTSCORE
        else None
    ),
    "bertscore_use_fast_tokenizer": (
        BERTSCORE_USE_FAST_TOKENIZER
        if RUN_BERTSCORE
        else None
    ),
    "bertscore_fallback_multilingual": (
        BERTSCORE_FALLBACK_MULTILINGUAL
        if RUN_BERTSCORE
        else None
    ),
    "metric_labels": METRIC_LABELS,
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "evaluation_config.json",
    ),
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Exported:")
for path in [
    per_sample_path,
    overall_path,
    overall_wide_path,
    per_type_path if not per_type_summary.empty else None,
]:
    if path is not None:
        print(" -", path)


## 16. Final summary

In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================
print("=" * 110)
print("FINAL SUMMARY — VIETFINQA LOCAL EVALUATION")
print("=" * 110)
print("ROUGE backend          : rouge_score.RougeScorer")
print("BERTScore backend       : bert_score.score")
print(f"BERTScore model         : {BERTSCORE_MODEL}")
print(f"Prediction files used   : {len(resolved_model_files)}")
print(f"Total evaluated rows    : {len(df_scores):,}")
print("Summary method          : arithmetic mean over all rows in each file")

print("\nFILES USED")
for model_name, file_path in resolved_model_files.items():
    row_count = int(
        (df_scores["model"] == model_name).sum()
    )
    print(
        f"- {model_name}: "
        f"{os.path.basename(file_path)} "
        f"({row_count:,} rows)"
    )

if missing_model_files:
    print("\nFILES NOT FOUND / SKIPPED")
    for model_name, file_name in missing_model_files.items():
        print(f"- {model_name}: {file_name}")

SUMMARY_METRICS = list(METRIC_LABELS.values())

summary_columns = [
    column
    for column in SUMMARY_METRICS
    if column in overall_wide.columns
]
final_summary_table = (
    overall_wide[summary_columns]
    .copy()
    .round(4)
)

print("\nOVERALL SUMMARY")
display(final_summary_table)

best_rows = []
for metric, label in METRIC_LABELS.items():
    metric_rows = overall_summary[
        overall_summary["metric"] == metric
    ].dropna(subset=["mean"])

    if metric_rows.empty:
        continue

    if HIGHER_IS_BETTER[metric]:
        best_index = metric_rows["mean"].idxmax()
        direction = "highest"
    else:
        best_index = metric_rows["mean"].idxmin()
        direction = "lowest"

    best = metric_rows.loc[best_index]
    best_rows.append({
        "metric": label,
        "best_model": best["model"],
        "best_mean": float(best["mean"]),
        "selection": direction,
    })

best_models_summary = pd.DataFrame(best_rows)
print("\nBEST METHOD BY METRIC")
display(
    best_models_summary.round({
        "best_mean": 4,
    })
)

bert_ranking = overall_summary[
    overall_summary["metric"] == "bertscore_f1"
][["model", "mean"]].dropna().sort_values(
    "mean",
    ascending=False,
)

print("\nPHOBERT BERTSCORE-F1 RANKING")
if bert_ranking.empty:
    print(
        "BERTScore was not available. "
        "Check model download, tokenizer/model compatibility, GPU memory, "
        "and RUN_BERTSCORE."
    )
else:
    bert_ranking = bert_ranking.rename(
        columns={"mean": "bertscore_f1"}
    )
    bert_ranking.insert(
        0,
        "rank",
        range(1, len(bert_ranking) + 1),
    )
    display(bert_ranking.round(4))

final_summary_csv = os.path.join(
    OUTPUT_DIR,
    "final_summary.csv",
)
best_models_csv = os.path.join(
    OUTPUT_DIR,
    "best_method_by_metric.csv",
)
bert_ranking_csv = os.path.join(
    OUTPUT_DIR,
    "phobert_bertscore_ranking.csv",
)
final_summary_txt = os.path.join(
    OUTPUT_DIR,
    "final_summary.txt",
)

final_summary_table.to_csv(
    final_summary_csv,
    encoding="utf-8-sig",
)
best_models_summary.to_csv(
    best_models_csv,
    index=False,
    encoding="utf-8-sig",
)
bert_ranking.to_csv(
    bert_ranking_csv,
    index=False,
    encoding="utf-8-sig",
)

with open(
    final_summary_txt,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "FINAL SUMMARY — VIETFINQA LOCAL EVALUATION\n"
    )
    file.write("=" * 80 + "\n")
    file.write("ROUGE backend: rouge_score.RougeScorer\n")
    file.write(
        "BERTScore backend: "
        "bert_score.score\n"
    )
    file.write(
        f"BERTScore model: {BERTSCORE_MODEL}\n"
    )
    file.write(
        f"Prediction files used: "
        f"{len(resolved_model_files)}\n"
    )
    file.write(
        f"Total evaluated rows: "
        f"{len(df_scores):,}\n\n"
    )

    file.write("OVERALL SUMMARY\n")
    file.write(
        final_summary_table.to_string()
        + "\n\n"
    )

    file.write("BEST METHOD BY METRIC\n")
    file.write(
        best_models_summary.to_string(index=False)
        + "\n\n"
    )

    file.write("PHOBERT BERTSCORE-F1 RANKING\n")
    file.write(
        bert_ranking.to_string(index=False)
        + "\n"
    )

print("\nFINAL SUMMARY FILES")
for path in [
    final_summary_csv,
    best_models_csv,
    bert_ranking_csv,
    final_summary_txt,
]:
    print("-", path)

print("=" * 110)
